# Chapter 11 — Python Control

WebApp Lua 프로그래밍이나 ROS2 CLI(`ros2 service call`)의 번거로운 문법 대신, Python 코드로 `/fairino_remote_command_service`를 한 줄씩 호출해 로봇을 제어합니다.

핵심은 `send_cmd()` 헬퍼 함수 하나로 압축한 것 — Jupyter Notebook은 그 결과를 셀 단위로 바로 확인하기 위한 실행환경일 뿐입니다.

**사전 준비**
- 터미널에서 `ros2 run fairino_hardware_v3_9_7 ros2_cmd_server` 가 실행 중이어야 합니다 (로봇당 반드시 1개 프로세스만 — 중복 실행 시 소리 없이 충돌합니다)
- ROS2(Jazzy) 환경을 source한 뒤 이 노트북을 그 환경의 커널로 실행하세요

In [16]:
import rclpy
from rclpy.node import Node
from fairino_msgs.srv import RemoteCmdInterface

rclpy.init()
node = Node('fr5_python_control')
client = node.create_client(RemoteCmdInterface, '/fairino_remote_command_service')

if not client.wait_for_service(timeout_sec=5.0):
    raise RuntimeError("서비스에 연결할 수 없습니다. ros2_cmd_server가 실행 중인지 확인하세요.")

print("연결 성공")

연결 성공


## send_cmd() — 화이트리스트 기반 안전 호출

`ALLOWED_COMMANDS`는 Chapter 10에서 실기로 **정상 작동이 확인된** 명령어만 담고 있습니다.

`MoveL` / `MoveC` / `StartJOG` / `StopJOG` / `ImmStopJOG`처럼 서버 크래시(세그폴트)가 확정된 명령어는 목록에 아예 없기 때문에, 함수 이름 검사 단계에서 로봇으로 전송되기 전에 거부됩니다.

In [17]:
import re

# Chapter 10 API Description 실기 테스트에서 정상 작동 확인된 명령어만 포함
ALLOWED_COMMANDS = {
    "JNTPoint", "CARTPoint", "GET",
    "RobotEnable", "Mode", "SetSpeed", "DragTeachSwitch",
    "SetToolCoord", "SetWObjCoord", "SetLoadWeight", "SetLoadCoord",
    "SetRobotInstallPos", "SetRobotInstallAngle",
    "MoveJ", "StopMotion",
    "SetDO", "SetToolDO", "SetAO", "SetToolAO",
    "ActGripper", "MoveGripper",
    "SetCollisionStrategy", "SetAnticollision", "SetPowerLimit",
    "ResetAllError",
}


def send_cmd(cmd_str: str, timeout_sec: float = 5.0) -> str:
    """cmd_str 예: 'SetSpeed(10)'. 화이트리스트에 없는 함수명은 로봇에 보내지 않고 즉시 예외를 발생시킨다."""
    match = re.match(r'\s*([A-Za-z_][A-Za-z0-9_]*)\s*\(', cmd_str)
    if not match:
        raise ValueError(f"명령어 형식이 아닙니다: {cmd_str!r}")

    func_name = match.group(1)
    if func_name not in ALLOWED_COMMANDS:
        raise ValueError(
            f"'{func_name}'은(는) 허용되지 않은 명령어입니다 (크래시 위험 또는 미검증).\n"
            f"허용 목록: {sorted(ALLOWED_COMMANDS)}"
        )

    request = RemoteCmdInterface.Request()
    request.cmd_str = cmd_str
    future = client.call_async(request)
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)

    if future.result() is None:
        raise TimeoutError(f"응답 시간 초과: {cmd_str}")

    return future.result().cmd_res

**주의 — `GET()`은 함수명은 화이트리스트에 있지만 인자에 따라 여전히 위험합니다.**

저장된 적 없는 id를 조회하면(`GET(JNT,0)` 등) 벤더 라이브러리(`libfairino.so`)의 버그로 서버가 double free로 죽습니다. **`JNTPoint`/`CARTPoint`로 미리 저장해 둔 id만 `GET()`으로 조회하세요.**

## 실습 — Chapter 10에서 검증된 시퀀스를 한 줄씩 실행

아래부터는 명령어 하나당 셀 하나씩입니다. 순서대로 실행하면서 WebApp 화면(Speed 표시, Enable 상태 등)이 실시간으로 바뀌는 걸 같이 확인하세요.

**속도 설정**

In [18]:
send_cmd("SetSpeed(10)")

'0'

**로봇 활성화 (Enable)**

In [19]:
send_cmd("RobotEnable(1)")

'0'

**모드 설정**

In [20]:
send_cmd("Mode(0)")

'0'

### 현재 관절각(J1~J6) 읽어오기

`JNTPoint`에 임의의 값(예: 전부 0)을 넣고 `MoveJ`로 이동하면, 지금 로봇 자세와 크게 다를 경우 로봇이 갑자기 크게 움직일 수 있습니다.

안전하게 첫 실습을 하려면, **로봇이 지금 서 있는 자세 그대로**를 포인트로 저장해서 "제자리로 이동"부터 확인하는 게 좋습니다. `/nonrt_state_data` 토픽(`fairino_msgs/msg/RobotNonrtState`)을 구독하면 `j1_cur_pos`~`j6_cur_pos` 필드로 실시간 관절각을 읽을 수 있습니다.

In [21]:
from fairino_msgs.msg import RobotNonrtState

def read_current_joints():
    """현재 J1~J6 관절각(도)을 튜플로 반환. /nonrt_state_data를 잠깐 구독해서 1개 메시지만 받고 정리한다."""
    latest = {}

    def _on_state(msg):
        latest['msg'] = msg

    sub = node.create_subscription(RobotNonrtState, '/nonrt_state_data', _on_state, 1)
    for _ in range(50):
        rclpy.spin_once(node, timeout_sec=0.1)
        if 'msg' in latest:
            break
    node.destroy_subscription(sub)

    if 'msg' not in latest:
        raise TimeoutError("nonrt_state_data 메시지를 받지 못했습니다. ros2_cmd_server/로봇 연결 상태를 확인하세요.")

    s = latest['msg']
    return (s.j1_cur_pos, s.j2_cur_pos, s.j3_cur_pos, s.j4_cur_pos, s.j5_cur_pos, s.j6_cur_pos)

In [22]:
j1, j2, j3, j4, j5, j6 = read_current_joints()
print(f"현재 관절각: J1={j1:.2f} J2={j2:.2f} J3={j3:.2f} J4={j4:.2f} J5={j5:.2f} J6={j6:.2f}")

현재 관절각: J1=-90.00 J2=-90.00 J3=90.00 J4=-90.00 J5=-90.00 J6=0.00


**현재 자세로 포인트 저장 (id=1)**

In [23]:
# 위에서 읽은 현재 관절값을 그대로 id=1로 저장 → MoveJ 실행해도 로봇이 '제자리'로만 이동함(안전한 첫 실습)
send_cmd(f"JNTPoint(1,{j1},{j2},{j3},{j4},{j5},{j6})")

'0'

**저장한 포인트로 이동 (MoveJ)**

point_name은 `JNT`+번호 형식의 문자열이어야 합니다(숫자만 넣으면 에러). `MoveJ(point_name, vel, tool, user)`

In [24]:
send_cmd("MoveJ(JNT1,10,1,0)")

'0'

**그리퍼 활성화**

In [25]:
send_cmd("ActGripper(1,1)")

'0'

**그리퍼 절반 닫기**

In [26]:
send_cmd("MoveGripper(1,50,30,30,2000,0,0,0,0,0)")

'0'

## 조인트별 개별 제어 (Soft Limit)

**Soft Limit이란**: 각 조인트가 소프트웨어적으로 회전할 수 있는 각도 범위입니다. 하드웨어 물리 한계(Hard Limit)보다 몇 도 안쪽으로 설정되어 있어서, 케이블이 꼬이거나 팔이 자기 자신과 부딪히기 전에 미리 멈추게 해줍니다.

**비유**: 사람 팔꿈치가 180도 넘게 뒤로 안 꺾이는 것과 비슷합니다 — 관절 구조상 갈 수 있는 한계 전에 안전하게 멈추는 지점을 미리 정해둔 것.

**Chapter 1에서 확인된 값**

| Joint | Soft Limit |
|---|---|
| J1 | ±175° |
| J2 | +85° ~ -265° |
| J3 | ±150° |
| J4 | +85° ~ -265° |
| J5 | ±175° |
| J6 | ±175° |

아래 `jog_joint()`는 조인트 하나만 현재 위치에서 상대 이동시키고, 목표각이 이 표를 벗어나면 `send_cmd()`(로봇 전송)까지 가지 않고 파이썬 단에서 미리 막습니다 — 화이트리스트가 "함수 이름"을 걸러낸다면, 이건 "값의 범위"를 걸러내는 두 번째 안전장치입니다.

In [27]:
# Chapter 1에서 확인된 Soft Limit (도)
SOFT_LIMITS = {
    1: (-175, 175),
    2: (-265, 85),
    3: (-150, 150),
    4: (-265, 85),
    5: (-175, 175),
    6: (-175, 175),
}

JOG_POINT_ID = 2  # id=1은 위에서 저장한 '홈 포지션'이라 건드리지 않고 별도 id 사용


def jog_joint(joint_num: int, delta_deg: float, vel: int = 10) -> float:
    """joint_num(1~6)을 현재 위치에서 delta_deg(도)만큼 상대 이동. Soft Limit을 벗어나면 로봇에 보내지 않고 예외를 발생시킨다."""
    if joint_num not in SOFT_LIMITS:
        raise ValueError("joint_num은 1~6 사이여야 합니다.")

    joints = list(read_current_joints())
    idx = joint_num - 1
    target = joints[idx] + delta_deg
    lo, hi = SOFT_LIMITS[joint_num]

    if not (lo <= target <= hi):
        raise ValueError(
            f"J{joint_num} 목표값 {target:.2f}°가 Soft Limit 범위({lo}°~{hi}°)를 벗어납니다."
        )

    joints[idx] = target
    send_cmd(f"JNTPoint({JOG_POINT_ID},{joints[0]},{joints[1]},{joints[2]},{joints[3]},{joints[4]},{joints[5]})")
    send_cmd(f"MoveJ(JNT{JOG_POINT_ID},{vel},1,0)")
    print(f"J{joint_num} 이동: {target - delta_deg:.2f}° → {target:.2f}°")
    return target

**J1을 현재 위치에서 +10° 회전**

In [30]:
jog_joint(1, 10)

J1 이동: -89.99° → -79.99°


-79.99456024169922

**Soft Limit 차단 데모 — 명백히 범위를 벗어나는 이동은 로봇에 전달되지 않음**

In [31]:
try:
    jog_joint(2, -400)  # J2 Soft Limit(-265~85)을 확실히 벗어나는 큰 값
except ValueError as e:
    print("차단됨:", e)

차단됨: J2 목표값 -490.00°가 Soft Limit 범위(-265°~85°)를 벗어납니다.


## 차단 데모 — 위험 명령어는 로봇에 도달하지 못하고 파이썬 단에서 거부됨

In [32]:
try:
    send_cmd("MoveL(JNT1,10,1,0)")
except ValueError as e:
    print("차단됨:", e)

차단됨: 'MoveL'은(는) 허용되지 않은 명령어입니다 (크래시 위험 또는 미검증).
허용 목록: ['ActGripper', 'CARTPoint', 'DragTeachSwitch', 'GET', 'JNTPoint', 'Mode', 'MoveGripper', 'MoveJ', 'ResetAllError', 'RobotEnable', 'SetAO', 'SetAnticollision', 'SetCollisionStrategy', 'SetDO', 'SetLoadCoord', 'SetLoadWeight', 'SetPowerLimit', 'SetRobotInstallAngle', 'SetRobotInstallPos', 'SetSpeed', 'SetToolAO', 'SetToolCoord', 'SetToolDO', 'SetWObjCoord', 'StopMotion']


## 정리 — 로봇 비활성화 후 노드 종료

**로봇 비활성화 (Disable)**

In [33]:
send_cmd("RobotEnable(0)")

'0'

In [34]:
node.destroy_node()
rclpy.shutdown()